# Week 53, Day 5: Knapsack

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w053d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
items = [(60, 1), (100, 2), (120, 3)]     # (value, weight)
capacity = 5

best = [0] * (capacity + 1)
for value, weight in items:
    for c in range(capacity, weight - 1, -1):
        best[c] = max(best[c], best[c - weight] + value)
print(best)

again = [0] * (capacity + 1)
for value, weight in items:
    for c in range(weight, capacity + 1):
        again[c] = max(again[c], again[c - weight] + value)
print(again)

In [ ]:
def knapsack(items, capacity):
    """items: (value, weight). Returns the best value and the indices taken."""
    n = len(items)
    best = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        value, weight = items[i - 1]
        for c in range(capacity + 1):
            best[i][c] = best[i - 1][c]
            if weight <= c:
                best[i][c] = max(best[i][c], best[i - 1][c - weight] + value)
    taken, c = [], capacity
    for i in range(n, 0, -1):
        if best[i][c] != best[i - 1][c]:
            taken.append(i - 1)
            c -= items[i - 1][1]
    return best[n][capacity], taken[::-1]

print(knapsack([(60, 10), (100, 20), (120, 30)], 50))

In [ ]:
def bundles(value, weight, copies):
    """Split k copies into 1, 2, 4, ... and a remainder, as 0/1 items."""
    out, size = [], 1
    while copies > 0:
        take = min(size, copies)
        out.append((value * take, weight * take))
        copies -= take
        size *= 2
    return out

print(bundles(5, 3, 13))

In [ ]:
def reachable_sums(numbers):
    reach = 1                      # bit 0: the empty subset sums to 0
    for x in numbers:
        reach |= reach << x
    return [c for c in range(sum(numbers) + 1) if reach >> c & 1]

print(reachable_sums([3, 5, 6]))

> **Key idea.** The 0/1 knapsack table best(i, c) takes or leaves item i − 1: max(best(i − 1, c), best(i − 1, c − w) + v), filled row by row in O(n × W), with the items found by walking back up. One row suffices if you loop capacities downwards; upwards gives the unbounded knapsack. Bounded copies split into bundles of 1, 2, 4, ... Subset sum, partition, counting subsets and target signs are knapsacks without values. The DP is pseudo-polynomial: for huge capacities, use a table by value, branch and bound, or meet in the middle.

### Exercise: Packing the van

Write `best_load(items, capacity)` for 0/1 items `(name, value, weight)`: `(value, names)`, the most value within the capacity and the sorted names of one best choice, from the full table walked back. Then `best_value(items, capacity)`: the value only, with **one row**.

In [ ]:
def best_load(items, capacity):
    ___

def best_value(items, capacity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(items, cap):\n    best = 0\n    for r in range(len(items) + 1):\n        for combo in itertools.combinations(items, r):\n            if sum(w for _, _, w in combo) <= cap:\n                best = max(best, sum(v for _, v, _ in combo))\n    return best\ndef test_load():\n    \"0/1\"\n    items = [(\"tent\", 60, 10), (\"stove\", 100, 20), (\"chairs\", 120, 30)]\n    assert best_load(items, 50) == (220, [\"chairs\", \"stove\"]) and best_value(items, 50) == 220\n    assert best_load([], 10) == (0, []) and best_load(items, 0) == (0, [])\n    rng = random.Random(1)\n    for _ in range(80):\n        items = [(f\"i{k}\", rng.randint(1, 20), rng.randint(1, 10)) for k in range(rng.randint(0, 8))]\n        cap = rng.randint(0, 30)\n        value, names = best_load(items, cap)\n        lookup = {n: (v, w) for n, v, w in items}\n        assert value == brute(items, cap) == best_value(items, cap)\n        assert sum(lookup[n][0] for n in names) == value and sum(lookup[n][1] for n in names) <= cap\n        assert names == sorted(set(names))\n    big = [(str(k), (k * 37) % 100 + 1, (k * 17) % 50 + 1) for k in range(200)]\n    assert best_value(big, 2000) == best_load(big, 2000)[0]")

### Exercise: Cutting the rod

A rod of length n can be cut into whole-number pieces; `prices[k]` is the price of a piece of length k + 1. Write `best_cut(prices, n)`: `(money, pieces)`, the most money and a list of piece lengths (sorted, longest first), as an unbounded knapsack with one row and a "last piece" list for reconstruction.

In [ ]:
def best_cut(prices, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from functools import cache\ndef test_rod():\n    \"unbounded\"\n    prices = [1, 5, 8, 9, 10, 17, 17, 20]\n    money, pieces = best_cut(prices, 8)\n    assert money == 22 and sum(pieces) == 8 and sum(prices[p - 1] for p in pieces) == 22\n    assert best_cut(prices, 0) == (0, []) and best_cut([3], 4) == (12, [1, 1, 1, 1])\n    @cache\n    def ref(n, prices):\n        return max([0] + [prices[k] + ref(n - k - 1, prices) for k in range(min(n, len(prices)))])\n    for n in range(0, 25):\n        for prices in ((1, 5, 8, 9, 10, 17, 17, 20), (2, 3), (1, 1, 10), (3, 5, 8, 9, 10, 17, 17, 20)):\n            money, pieces = best_cut(list(prices), n)\n            assert money == ref(n, prices) and sum(prices[p - 1] for p in pieces) == money\n            assert sum(pieces) <= n and pieces == sorted(pieces, reverse=True)")

### Exercise: Fair shares

Two hikers share out bags of given (positive, whole-number) weights. Write `fairest_split(weights)`: the smallest possible difference between the two hikers' totals, using an integer as a bitset of reachable sums. Then `can_halve(weights)`: True if they can carry exactly equal weights.

In [ ]:
def fairest_split(weights):
    ___

def can_halve(weights):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(ws):\n    total = sum(ws)\n    best = total\n    for picks in itertools.product([0, 1], repeat=len(ws)):\n        s = sum(w for w, p in zip(ws, picks) if p)\n        best = min(best, abs(total - 2 * s))\n    return best\ndef test_split():\n    \"partition\"\n    assert fairest_split([1, 6, 11, 5]) == 1 and can_halve([1, 5, 11, 5]) and not can_halve([1, 2, 3, 5])\n    assert fairest_split([]) == 0 and can_halve([]) and fairest_split([7]) == 7\n    rng = random.Random(2)\n    for _ in range(150):\n        ws = [rng.randint(1, 20) for _ in range(rng.randint(0, 10))]\n        assert fairest_split(ws) == brute(ws) and can_halve(ws) == (brute(ws) == 0)\n    big = [rng.randint(1, 1000) for _ in range(300)]\n    assert fairest_split(big) <= 1")

### Exercise: Plus or minus

Write `sign_ways(numbers, target)`: how many ways to put `+` or `-` before each number (whole numbers, 0 or more) so the total is the target. Turn it into a counting knapsack: the plus group must sum to (total + target) / 2, which must be a whole number between 0 and the total. Note that a 0 can take either sign.

In [ ]:
def sign_ways(numbers, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(nums, target):\n    return sum(1 for signs in itertools.product([1, -1], repeat=len(nums)) if sum(s * x for s, x in zip(signs, nums)) == target)\ndef test_signs():\n    \"signs\"\n    assert sign_ways([1, 1, 1, 1, 1], 3) == 5 and sign_ways([1], 2) == 0 and sign_ways([], 0) == 1\n    assert sign_ways([0, 0, 1], 1) == 4 and sign_ways([2, 3], 0) == 0\n    rng = random.Random(3)\n    for _ in range(200):\n        nums = [rng.randint(0, 6) for _ in range(rng.randint(0, 9))]\n        t = rng.randint(-12, 12)\n        assert sign_ways(nums, t) == brute(nums, t), (nums, t)\n    assert sign_ways([1] * 60, 0) == 118264581564861424")

### Exercise: A shop with limited stock

A shop has `copies` of each item `(value, weight, copies)`. Write `bundles(value, weight, copies)`: the binary-split bundles as `(value, weight)` 0/1 items (sizes 1, 2, 4, ..., then the remainder), and `best_bounded(items, capacity)`: the most value, by running a one-row 0/1 knapsack over all the bundles.

In [ ]:
def bundles(value, weight, copies):
    ___

def best_bounded(items, capacity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(items, cap):\n    best = 0\n    for counts in itertools.product(*[range(k + 1) for _, _, k in items]):\n        if sum(c * w for c, (_, w, _) in zip(counts, items)) <= cap:\n            best = max(best, sum(c * v for c, (v, _, _) in zip(counts, items)))\n    return best\ndef test_bounded():\n    \"bounded\"\n    assert bundles(5, 3, 13) == [(5, 3), (10, 6), (20, 12), (30, 18)]\n    assert bundles(1, 1, 0) == [] and bundles(2, 1, 1) == [(2, 1)] and len(bundles(1, 1, 1000)) == 10\n    for k in range(0, 40):\n        sizes = [w for _, w in bundles(1, 1, k)]\n        sums = {sum(c) for r in range(len(sizes) + 1) for c in itertools.combinations(sizes, r)}\n        assert sums == set(range(k + 1)), k\n    rng = random.Random(4)\n    for _ in range(60):\n        items = [(rng.randint(1, 15), rng.randint(1, 8), rng.randint(0, 4)) for _ in range(rng.randint(0, 4))]\n        cap = rng.randint(0, 30)\n        assert best_bounded(items, cap) == brute(items, cap)\n    assert best_bounded([(3, 2, 10 ** 6), (5, 3, 10 ** 6)], 10_000) == 16_666")

### Bug hunt: One tent, packed twice

This one-row 0/1 knapsack loops the capacity **upwards**, so an item already added at a smaller capacity is added again: with one item of value 10 and weight 2, and capacity 6, it prints `30`. Loop downwards so it prints `10`.

In [ ]:
items = [(10, 2)]
capacity = 6
best = [0] * (capacity + 1)
for value, weight in items:
    for c in range(weight, capacity + 1):
        best[c] = max(best[c], best[c - weight] + value)
print(best[capacity])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"each item once\"\n    expect_output(\"10\")")

### Bug hunt: Nothing to count

This counts the subsets of `[1, 2, 3]` summing to 3 (`{3}` and `{1, 2}`), but `count[0]` starts at 0, so no subset is ever counted and it prints `0`. The empty subset sums to 0: start `count[0]` at 1 so it prints `2`.

In [ ]:
numbers = [1, 2, 3]
goal = 3
count = [0] * (goal + 1)
for x in numbers:
    for c in range(goal, x - 1, -1):
        count[c] += count[c - x]
print(count[goal])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the empty subset\"\n    expect_output(\"2\")")

### Bug hunt: Rows are not items

Row i of the table is about the first i items, so a change between rows i − 1 and i means item **i − 1** was taken. This walk back records the row number instead, and prints `[2, 3]`, which isn't even a valid index for the last one. Record the item's index so it prints `[1, 2]`.

In [ ]:
items = [(60, 10), (100, 20), (120, 30)]
capacity = 50
n = len(items)
best = [[0] * (capacity + 1) for _ in range(n + 1)]
for i in range(1, n + 1):
    value, weight = items[i - 1]
    for c in range(capacity + 1):
        best[i][c] = best[i - 1][c]
        if weight <= c:
            best[i][c] = max(best[i][c], best[i - 1][c - weight] + value)
taken, c = [], capacity
for i in range(n, 0, -1):
    if best[i][c] != best[i - 1][c]:
        taken.append(i)
        c -= items[i - 1][1]
print(sorted(taken))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"row above\"\n    expect_output(\"[1, 2]\")")

### Puzzle: Subset sum with booleans

Arrange the lines into a one-row subset-sum check. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the 0/1 best for items (value, weight) (6, 1), (10, 2), (12, 3) with capacity 5
q = ___   # the capacity loop direction for the unbounded knapsack: "up" or "down"
r = ___   # the bundle sizes for 10 copies, as a list
s = ___   # the reachable sums of [2, 3] as a sorted list, including 0
t = ___   # True or False: knapsack DP is polynomial in the number of digits of the capacity

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 22 and q == \"up\" and r == [1, 2, 4, 3], \"p, q and r\"\n    assert s == [0, 2, 3, 5] and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: weight and space

An expedition's pack animal can carry at most `max_weight` kilograms **and** `max_volume` litres. Each item `(name, value, weight, volume)` is taken at most once. Write `best_pack(items, max_weight, max_volume)`: `(value, names)`, the most value within both limits and one best choice (sorted names). The state needs both budgets: best(i, w, v). Use a full three-dimensional table so you can walk back, or a two-dimensional one per item with a record of the choices.

Then `best_pack_value(items, max_weight, max_volume)`: the value only, with a single two-dimensional table updated with **both** loops running downwards.

In [ ]:
def best_pack(items, max_weight, max_volume):
    ___

def best_pack_value(items, max_weight, max_volume):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(items, W, V):\n    best = 0\n    for r in range(len(items) + 1):\n        for combo in itertools.combinations(items, r):\n            if sum(i[2] for i in combo) <= W and sum(i[3] for i in combo) <= V:\n                best = max(best, sum(i[1] for i in combo))\n    return best\ndef test_pack():\n    \"two budgets\"\n    items = [(\"rope\", 10, 2, 3), (\"food\", 30, 6, 4), (\"water\", 25, 5, 5), (\"tent\", 40, 7, 9)]\n    value, names = best_pack(items, 13, 12)\n    assert value == brute(items, 13, 12) == best_pack_value(items, 13, 12) == 65 and names == [\"food\", \"rope\", \"water\"]\n    assert best_pack([], 5, 5) == (0, []) and best_pack(items, 0, 100) == (0, [])\n    rng = random.Random(5)\n    for _ in range(60):\n        items = [(f\"x{k}\", rng.randint(1, 20), rng.randint(1, 6), rng.randint(1, 6)) for k in range(rng.randint(0, 7))]\n        W, V = rng.randint(0, 15), rng.randint(0, 15)\n        value, names = best_pack(items, W, V)\n        lookup = {i[0]: i for i in items}\n        assert value == brute(items, W, V) == best_pack_value(items, W, V)\n        assert sum(lookup[n][1] for n in names) == value\n        assert sum(lookup[n][2] for n in names) <= W and sum(lookup[n][3] for n in names) <= V")

## Warm-up

The only difference is the direction of the inner loop. Backwards, each item is used at most once (the 0/1 knapsack: 220 from the second and third items). Forwards, an item can be used again and again (the unbounded knapsack: five copies of the first item make 300). Today explains why.

## The 0/1 table

Items have values and weights; take each item **at most once**, total weight at most W, most value. The state is how many items you've considered and how much capacity you have:

**best(i, c) = the most value using only the first i items with capacity c**.

Item i − 1 is either left out or taken (if it fits):

best(i, c) = max(best(i − 1, c), best(i − 1, c − wᵢ) + vᵢ),

with best(0, c) = 0. Fill row by row (i upwards); each row reads only the row before. The answer is best(n, W). There are (n + 1) × (W + 1) entries, each O(1), so O(n × W) time.

To find **which** items, walk back from (n, W): if best(i, c) equals best(i − 1, c), item i − 1 wasn't needed; otherwise it was taken, so subtract its weight and move up a row.

## One row, backwards

Row i reads only row i − 1, so keep one list `best[c]`. The catch is the order. The update `best[c] = max(best[c], best[c - w] + v)` needs `best[c - w]` from the **previous** row (the item not yet used). Looping c **downwards**, the smaller index c − w hasn't been overwritten yet in this pass, so it's still the previous row's value. Looping upwards, `best[c - w]` may already include this item, and the item gets taken again.

That "bug" is exactly the **unbounded knapsack**, where every item can be used any number of times: there the recurrence is best(c) = max over items of best(c − wᵢ) + vᵢ, which wants this row's values. So:

- 0/1 knapsack (each item once): capacity loop **downwards**.
- Unbounded (unlimited copies): capacity loop **upwards**.

This one-row version is O(W) memory, but loses the reconstruction (as on Tuesday). Keep the full table, or a table of booleans recording "taken at (i, c)", when you need the items.

## Unbounded and bounded

**Unbounded** problems: cutting a rod into pieces with prices for each length, fewest coins (Tuesday), making change. Loop the capacity upwards, or think of it as one-dimensional: best(c) = max over items of best(c − w) + v.

**Bounded**: item i has kᵢ copies. Treating each copy as a separate 0/1 item works but costs O(W × total copies). **Binary splitting** is faster: split k copies into bundles of 1, 2, 4, ..., and a remainder (13 = 1 + 2 + 4 + 6). Every number of copies from 0 to k is a sum of some bundles, so treating each bundle as one 0/1 item gives the same answers with only about log k items per original item.

## Knapsacks in disguise

Many problems are knapsacks with the values removed:

- **Subset sum**: can some of these numbers add up to T? A table of booleans: `can[c]` is True if some subset sums to c. Update `can[c] = can[c] or can[c - x]`, looping downwards.
- **Fair partition**: split numbers into two groups with sums as close as possible: find the reachable subset sum nearest to half the total.
- **Counting**: how many subsets sum to T? `count[c] += count[c - x]`, downwards, with `count[0] = 1`.
- **Signs**: put + or − before each number to reach a target. If P is the sum of the plus group, P − (total − P) = target, so P = (total + target) / 2: a counting knapsack.

A neat Python trick for subset sums: an integer as a **bitset**, where bit c means "c is reachable". Adding a number x is `reach |= reach << x`, which handles every capacity at once in fast C code (bit shifts, from Week 55's toolkit).

## When the capacity is huge

O(n × W) looks polynomial, but W is a **number** in the input, written in about log W digits. Double the number of digits and W squares. That makes the knapsack DP **pseudo-polynomial**: fast when capacities are modest (thousands, millions), hopeless for W = 10¹⁸. The 0/1 knapsack is NP-hard in general, so no algorithm is known to be polynomial in the input's size.

When W is huge but **values** are small, swap the roles: min_weight(v) = the least weight that achieves value exactly v, a table of size n × (total value). When both are huge and n is modest, use Friday's branch and bound, or **meet in the middle**: list the subset sums of each half (2^(n/2) each), sort one, and binary-search it for each sum of the other.

## Practice

## Quiz

## Daily challenge

## Before you go

Name three problems from outside programming that are knapsacks in disguise (a budget, choices, and something to make as large as possible). Tomorrow is the Forge: a festival planner built from this week's DP patterns.